## Ingest stock data

In [0]:
import requests
import time
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone

API_KEY = "RVPR7AZ4VY4ZKUKX"

symbols = ["AAPL", "MSFT", "GOOGL", "TSLA"]

BASE_URL = "https://www.alphavantage.co/query"


In [0]:
def fetch_daily_data(symbol):
    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "outputsize": "compact",
        "apikey": API_KEY
    }
    
    response = requests.get(BASE_URL, params=params)
    data = response.json()
    
    if "Time Series (Daily)" not in data:
        raise Exception(f"Error fetching data for {symbol}: {data}")
    
    time_series = data["Time Series (Daily)"]
    
    records = []
    
    for date, values in time_series.items():
        records.append({
            "symbol": symbol,
            "trading_date": date,
            "open": float(values["1. open"]),
            "high": float(values["2. high"]),
            "low": float(values["3. low"]),
            "close": float(values["4. close"]),
            "volume": int(values["5. volume"]),
            "ingestion_timestamp": datetime.now(timezone.utc)
        })
    
    return records


In [0]:
all_data = []

for symbol in symbols:
    print(f"Fetching data for {symbol}")
    records = fetch_daily_data(symbol)
    all_data.extend(records)
    
    time.sleep(12)  # 5 requests/min (API allows 5 requests per minute, 60 / 5 = 12 seconds)


Fetching data for AAPL
Fetching data for MSFT
Fetching data for GOOGL
Fetching data for TSLA


In [0]:
df = spark.createDataFrame(all_data)

df = df.withColumn("trading_date", F.to_date("trading_date"))

#display(df.limit(10))
df.show()

+------+--------+--------------------+-------+-------+------+------------+--------+
| close|    high| ingestion_timestamp|    low|   open|symbol|trading_date|  volume|
+------+--------+--------------------+-------+-------+------+------------+--------+
|263.88|  266.29|2026-02-18 20:16:...| 255.54| 258.05|  AAPL|  2026-02-17|58469094|
|255.78|  262.23|2026-02-18 20:16:...| 255.45| 262.01|  AAPL|  2026-02-13|56290673|
|261.73|  275.72|2026-02-18 20:16:...| 260.18| 275.59|  AAPL|  2026-02-12|81077229|
| 275.5|  280.18|2026-02-18 20:16:...| 274.45|274.695|  AAPL|  2026-02-11|51931283|
|273.68|  275.37|2026-02-18 20:16:...| 272.94|274.885|  AAPL|  2026-02-10|34376898|
|274.62|   278.2|2026-02-18 20:16:...|  271.7|277.905|  AAPL|  2026-02-09|44623396|
|278.12| 280.905|2026-02-18 20:16:...|276.925| 277.12|  AAPL|  2026-02-06|50453414|
|275.91|   279.5|2026-02-18 20:16:...| 273.23| 278.13|  AAPL|  2026-02-05|52977441|
|276.49|  278.95|2026-02-18 20:16:...|272.285|272.285|  AAPL|  2026-02-04|90

In [0]:
# from delta.tables import DeltaTable

# table_name = "finance.stock_dlt.raw_stock_data"

# if spark.catalog.tableExists(table_name):

#     delta_table = DeltaTable.forName(spark, table_name)

#     (
#         delta_table.alias("target")
#         .merge(
#             df.alias("source"),
#             "target.symbol = source.symbol AND target.trading_date = source.trading_date"
#         )
#         .whenMatchedUpdateAll()
#         .whenNotMatchedInsertAll()
#         .execute()
#     )

# else:
#     (
#         df.write
#         .format("delta")
#         .mode("overwrite")
#         .saveAsTable(table_name)
#     )

(
    df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("finance.stock_dlt.raw_stock_data")
)

In [0]:
#display(spark.table("finance.stock_dlt.raw_stock_data"))
spark.table("finance.stock_dlt.raw_stock_data").show()


+------+--------+--------------------+-------+-------+------+------------+--------+
| close|    high| ingestion_timestamp|    low|   open|symbol|trading_date|  volume|
+------+--------+--------------------+-------+-------+------+------------+--------+
|263.88|  266.29|2026-02-18 20:16:...| 255.54| 258.05|  AAPL|  2026-02-17|58469094|
|255.78|  262.23|2026-02-18 20:16:...| 255.45| 262.01|  AAPL|  2026-02-13|56290673|
|261.73|  275.72|2026-02-18 20:16:...| 260.18| 275.59|  AAPL|  2026-02-12|81077229|
| 275.5|  280.18|2026-02-18 20:16:...| 274.45|274.695|  AAPL|  2026-02-11|51931283|
|273.68|  275.37|2026-02-18 20:16:...| 272.94|274.885|  AAPL|  2026-02-10|34376898|
|274.62|   278.2|2026-02-18 20:16:...|  271.7|277.905|  AAPL|  2026-02-09|44623396|
|278.12| 280.905|2026-02-18 20:16:...|276.925| 277.12|  AAPL|  2026-02-06|50453414|
|275.91|   279.5|2026-02-18 20:16:...| 273.23| 278.13|  AAPL|  2026-02-05|52977441|
|276.49|  278.95|2026-02-18 20:16:...|272.285|272.285|  AAPL|  2026-02-04|90

## Ingest Company

In [0]:
def fetch_company_overview(symbol):
    params = {
        "function": "OVERVIEW",
        "symbol": symbol,
        "apikey": API_KEY
    }

    response = requests.get(BASE_URL, params=params)
    data = response.json()

    if "Symbol" not in data:
        raise Exception(f"Error fetching overview for {symbol}: {data}")

    return {
        "symbol": data.get("Symbol"),
        "name": data.get("Name"),
        "sector": data.get("Sector"),
        "industry": data.get("Industry"),
        "exchange": data.get("Exchange"),
        "country": data.get("Country"),
        "market_cap": int(data.get("MarketCapitalization", 0)),
        "ingestion_timestamp": datetime.now(timezone.utc)
    }


In [0]:
company_data = []

for symbol in symbols:
    print(f"Fetching overview for {symbol}")
    company_data.append(fetch_company_overview(symbol))
    time.sleep(12)

company_df = spark.createDataFrame(company_data)

#display(company_df)
company_df.show()


Fetching overview for AAPL
Fetching overview for MSFT
Fetching overview for GOOGL
Fetching overview for TSLA
+-------+--------+--------------------+--------------------+-------------+--------------------+--------------------+------+
|country|exchange|            industry| ingestion_timestamp|   market_cap|                name|              sector|symbol|
+-------+--------+--------------------+--------------------+-------------+--------------------+--------------------+------+
|    USA|  NASDAQ|CONSUMER ELECTRONICS|2026-02-18 20:36:...|3878488637000|           Apple Inc|          TECHNOLOGY|  AAPL|
|    USA|  NASDAQ|SOFTWARE - INFRAS...|2026-02-18 20:36:...|2949613355000|Microsoft Corpora...|          TECHNOLOGY|  MSFT|
|    USA|  NASDAQ|INTERNET CONTENT ...|2026-02-18 20:36:...|3653536055000|Alphabet Inc Class A|COMMUNICATION SER...| GOOGL|
|    USA|  NASDAQ|  AUTO MANUFACTURERS|2026-02-18 20:37:...|1540861067000|           Tesla Inc|   CONSUMER CYCLICAL|  TSLA|
+-------+--------+-----

In [0]:
# company_table = "finance.stock_dlt.raw_company_info"

# if spark.catalog.tableExists(company_table):

#     delta_table = DeltaTable.forName(spark, company_table)

#     (
#         delta_table.alias("target")
#         .merge(
#             company_df.alias("source"),
#             "target.symbol = source.symbol"
#         )
#         .whenMatchedUpdateAll()
#         .whenNotMatchedInsertAll()
#         .execute()
#     )

# else:
#     (
#         company_df.write
#         .format("delta")
#         .mode("overwrite")
#         .saveAsTable(company_table)
#     )

company_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("finance.stock_dlt.raw_company_info")


In [0]:
# display(spark.table("finance.stock_dlt.raw_company_info"))
spark.table("finance.stock_dlt.raw_company_info").show()

+-------+--------+--------------------+--------------------+-------------+--------------------+--------------------+------+
|country|exchange|            industry| ingestion_timestamp|   market_cap|                name|              sector|symbol|
+-------+--------+--------------------+--------------------+-------------+--------------------+--------------------+------+
|    USA|  NASDAQ|CONSUMER ELECTRONICS|2026-02-18 19:01:...|3878488637000|           Apple Inc|          TECHNOLOGY|  AAPL|
|    USA|  NASDAQ|SOFTWARE - INFRAS...|2026-02-18 19:01:...|2949613355000|Microsoft Corpora...|          TECHNOLOGY|  MSFT|
|    USA|  NASDAQ|INTERNET CONTENT ...|2026-02-18 19:02:...|3653536055000|Alphabet Inc Class A|COMMUNICATION SER...| GOOGL|
|    USA|  NASDAQ|  AUTO MANUFACTURERS|2026-02-18 19:02:...|1540861067000|           Tesla Inc|   CONSUMER CYCLICAL|  TSLA|
|    USA|  NASDAQ|CONSUMER ELECTRONICS|2026-02-18 20:36:...|3878488637000|           Apple Inc|          TECHNOLOGY|  AAPL|
|    USA